In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
from parametric_impact_pinn_tf1 import SequentialParametricImpactPINN
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # CPU:-1; GPU0: 0

In [ ]:
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print('TensorFlow version: {}'.format(tf.__version__))

## 2. Define and run solver

In [ ]:
layers = [10, 64, 64, 64, 1]
n_impact = 6

# Normalization bounds for the 9 non-time input channels:
# [mx,  my,   k,    c,   D,   y0,   yt0,  x0,   xt0]
lb_params = [ 0.1,  0.1,  0.1,  0.0,  0.1, -5.0, -5.0, -5.0, -5.0]
ub_params = [10.0, 10.0, 10.0,  5.0, 10.0,  5.0,  5.0,  5.0,  5.0]

solver = SequentialParametricImpactPINN(
    mx=1.0,
    my=0.3,
    k=1.0,
    c=0.0,
    D=1.0,
    r=1.0,
    layers=layers,
    hyp_ini_weight_loss=(1.0, 1.0, 10.0),
    optimizer_LB=True,
)

results = solver.run(
    x0=[[0.0]],
    xt0=[[0.0]],
    y0=[[0.0]],
    yt0=[[-1.0]],
    n_impact=n_impact,
    T_vector=[1, 5, 2, 5, 4, 10],
    nIter_vector=[1000, 1000, 1000, 1000, 1000, 1000],
    hyp_ini_para_vector=[0.5, 4, 1, 4, 3, 5],
    num_time_points=1000,
    lb_params=lb_params,
    ub_params=ub_params,
)

## 3. Extract results

In [ ]:
all_time = results['time']
all_x    = results['x']
all_xt   = results['xt']
all_xtt  = results['xtt']
all_y    = results['y']
all_yt   = results['yt']
all_ytt  = results['ytt']

impact_times       = solver.impact_times          # per-segment durations
impact_time_stamps = np.cumsum(impact_times)      # cumulative, for plot markers

print('Learned impact times (per segment):', np.round(impact_times, 4))
print('Cumulative (PINN):                 ', np.round(impact_time_stamps, 4))

## 4. Plot — Deflection vs Time

In [ ]:
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Times New Roman'
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

fontsize_labels = 28
fontsize_ticks  = 28
fontsize_legend = 24
linewidth_plot  = 2
linewidth_vline = 1

plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x, label=r'$x(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_y, label=r'$y(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Deflection (m)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.8, 1.8)
plt.savefig('deflection_vs_time.png', format='png', dpi=300)
plt.show()

## 5. Plot — Velocity vs Time

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_xt, label=r'$\dot{x}(t)$', linewidth=linewidth_plot)
plt.plot(all_time, all_yt, label=r'$\dot{y}(t)$', linestyle='--', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Velocity (m/s)', fontsize=fontsize_labels, labelpad=12)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='upper right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('velocity_vs_time.png', format='png', dpi=300)
plt.show()

## 6. Plot — Relative Deflection (x - y)

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(all_time, all_x - all_y, label='PINN', linestyle='--', linewidth=linewidth_plot)
plt.axhline(y= 1.0, color='black', linestyle='-.', linewidth=2)
plt.axhline(y=-1.0, color='black', linestyle='-.', linewidth=2)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Relative Deflection (m)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend, loc='lower right')
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-1.4, 1.4)
plt.savefig('relative_deflection.png', format='png', dpi=300)
plt.show()

## 7. Plot — Impulse

In [ ]:
my = solver.my
delta_yt = np.diff(all_yt, axis=0)
time_for_delta_yt = all_time[:-1]
impulse = my * delta_yt

plt.figure(figsize=(10, 4.5))
plt.plot(time_for_delta_yt, impulse,
         label=r'$m(\dot{y}_{n+1} - \dot{y}_n)$', linewidth=linewidth_plot)
for t_imp in impact_time_stamps:
    plt.axvline(x=t_imp, color='gray', linestyle='--', linewidth=linewidth_vline)
plt.xlabel('Time (t)', fontsize=fontsize_labels, labelpad=8)
plt.ylabel('Impulse (Ns)', fontsize=fontsize_labels, labelpad=10)
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(nbins=10, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.tick_params(axis='both', labelsize=fontsize_ticks)
plt.legend(fontsize=fontsize_legend)
plt.tight_layout(pad=1.5)
plt.xlim(0, impact_time_stamps[-1] * 1.05)
plt.ylim(-0.6, 0.6)
plt.savefig('impulse_plot.png', format='png', dpi=300)
plt.show()

## 8. Plot — Impact Time Convergence per Segment

In [ ]:
for i, model in enumerate(solver.models):
    # lambda_1_log[k] has shape (1,1); extract scalar per iteration
    lambda_log = [float(v[0, 0]) for v in model.lambda_1_log]

    plt.figure(figsize=(4.5, 4.5))
    plt.plot(lambda_log, linewidth=2, label='PINN')
    plt.xlabel('Iteration', fontsize=24, labelpad=10)
    plt.ylabel('Impact time', fontsize=24, labelpad=10)
    ax = plt.gca()
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))
    ax.tick_params(axis='both', labelsize=24)
    plt.tight_layout(pad=1.5)
    plt.savefig(f'impact_convergence_{i+1}.png', format='png', dpi=300)
    plt.show()

## 9. Plot — Training Loss (last segment)

In [ ]:
# To inspect a different segment, change the index: solver.models[i]
model = solver.models[-1]

fig, ax = plt.subplots(figsize=(5, 2.7))
ax.loglog(model.loss_log,     label='Total loss')
ax.loglog(model.loss_icx_log, label='Loss_icx')
ax.loglog(model.loss_fx_log,  label='Loss_fx')
ax.loglog(model.loss_f_log,   label='Loss_f')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

## 10. True Parametric PINN — Train Once Across Many Parameter Cases

In [ ]:
"""
What makes this a TRUE parametric PINN:

- N parameter combinations are sampled via Latin Hypercube Sampling (LHS).
- ONE network per segment is trained on ALL N cases simultaneously.
  The physics loss (ODE residual + IC + impact condition) is enforced for
  every case in each gradient step — so the network learns a solution
  operator, not a single solution.
- After training, predictor.predict(mx, my, k, c, D) gives the full
  multi-impact response for any new parameter set instantly, with no
  retraining.
"""
from parametric_pinn_multi_case import train_parametric_pinn

# -------------------------------------------------------------------
# Parameter bounds for training  [mx, my, k, c, D, y0, yt0, x0, xt0]
# -------------------------------------------------------------------
lb_params_train = [0.5,  0.1,  0.5,  0.0,  0.5, -0.1, -1.5, -0.1, -0.1]
ub_params_train = [2.0,  0.6,  2.0,  0.1,  2.0,  0.1, -0.5,  0.1,  0.1]

# Fix ICs so only the physical parameters (mx, my, k, c, D) vary.
# Remove fixed_ic to also vary initial conditions.
fixed_ic = {'x0': 0.0, 'xt0': 0.0, 'y0': 0.0, 'yt0': -1.0}

predictor, seg_models = train_parametric_pinn(
    n_cases=50,                          # parameter cases per segment
    n_impacts=4,                         # number of impact segments
    lb_params=lb_params_train,
    ub_params=ub_params_train,
    r=1.0,                               # coefficient of restitution
    fixed_ic=fixed_ic,
    layers=[10, 64, 64, 64, 1],
    T_max_per_segment=[3.0, 5.0, 5.0, 8.0],
    nIter_per_segment=[3000, 3000, 3000, 3000],
    n_r=200,
    hyp_ini_para=0.5,
    hyp_ini_weight_loss=(1.0, 1.0, 10.0),
    optimizer_LB=True,
    seed=42,
)

## 11. Predict for New Parameter Sets — No Retraining

In [ ]:
# Three NEW parameter sets — none of these were training cases.
# The predictor uses the trained network + root-finding only.
test_cases = [
    dict(mx=1.0, my=0.3,  k=1.0,  c=0.0,  D=1.0,  label='Case A  (mx=1.0, my=0.3, k=1.0, D=1.0)'),
    dict(mx=1.5, my=0.5,  k=1.8,  c=0.05, D=0.7,  label='Case B  (mx=1.5, my=0.5, k=1.8, D=0.7)'),
    dict(mx=0.7, my=0.15, k=0.6,  c=0.0,  D=1.5,  label='Case C  (mx=0.7, my=0.15, k=0.6, D=1.5)'),
]

results_test = []
for tc in test_cases:
    res = predictor.predict(
        mx=tc['mx'], my=tc['my'], k=tc['k'], c=tc['c'], D=tc['D'],
        x0=0.0, xt0=0.0, y0=0.0, yt0=-1.0,
        num_points=500,
    )
    results_test.append(res)
    print(f"{tc['label']}")
    print(f"  Impact times: {np.round(res['impact_times'], 4)}")
    print()

## 12. Plot — Parametric Predictions for Three Different Parameter Sets

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=False)

for ax, res, tc in zip(axes, results_test, test_cases):
    t_end = res['impact_times'].sum()
    t_stamps = np.cumsum(res['impact_times'])

    ax.plot(res['time'], res['x'], label=r'$x(t)$ — main mass', linewidth=2)
    ax.plot(res['time'], res['y'], '--', label=r'$y(t)$ — impactor', linewidth=2)
    for ts in t_stamps:
        ax.axvline(x=ts, color='gray', linestyle=':', linewidth=1)
    ax.set_title(tc['label'], fontsize=14)
    ax.set_xlabel('Time', fontsize=13)
    ax.set_ylabel('Displacement', fontsize=13)
    ax.legend(fontsize=12)
    ax.set_xlim(0, t_end * 1.05)
    ax.tick_params(labelsize=12)

plt.suptitle('True Parametric PINN — Instant Prediction for New Parameters\n'
             '(no retraining, only forward pass + root-finding)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('parametric_predictions.png', dpi=150, bbox_inches='tight')
plt.show()